In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

In [5]:
customers_df = pd.read_csv("csv/Customers_members.csv")
products_df = pd.read_csv("csv/Products_overview.csv")
transactions_df = pd.read_csv("csv/Transactions_history.csv")

In [6]:
customers_df.head()

,CustomerID,AgeGroup,Region,LoyaltyLevel,MembershipYears
0,1,36-50,East,Bronze,9
1,2,26-35,East,Bronze,4
2,3,18-25,East,Silver,10
3,4,36-50,East,Bronze,14
4,5,26-35,North,Bronze,7


In [7]:
products_df.head()

,ProductID,Category,Brand,Price,Rating,PopularityScore
0,1,Electronics,TechMax,474.45,4.54,3
1,2,Gaming,SmartBuy,285.11,3.16,2
2,3,books,HomePro,370.79,3.01,1
3,4,Home,TechMax,416.38,3.38,1
4,5,Books,TechMax,62.37,3.42,1


In [8]:
transactions_df.head()

,CustomerID,ProductID,PurchaseCount,LastPurchaseDays,Spend
0,1,82,3,162,104.01
1,2,7,4,276,387.48
2,3,170,4,215,1507.40
3,3,9,1,251,341.40
4,4,64,3,354,1250.40


In [9]:
print("customers_df shape:", customers_df.shape)
print("products_df shape:", products_df.shape)
print("transactions_df shape:", transactions_df.shape)

customers_df shape: (1000, 5)
products_df shape: (200, 6)
transactions_df shape: (19166, 5)


In [10]:
display(customers_df.dtypes.rename("Data Type").to_frame())
display(products_df.dtypes.rename("Data Type").to_frame())
display(transactions_df.dtypes.rename("Data Type").to_frame())

,Data Type
CustomerID,int64
AgeGroup,str
Region,str
LoyaltyLevel,str
MembershipYears,int64


,Data Type
ProductID,int64
Category,str
Brand,str
Price,float64
Rating,float64
PopularityScore,int64


,Data Type
CustomerID,int64
ProductID,int64
PurchaseCount,int64
LastPurchaseDays,int64
Spend,float64


In [11]:
customers_df.isnull().sum().rename("Missing Values").to_frame()

,Missing Values
CustomerID,0
AgeGroup,0
Region,0
LoyaltyLevel,0
MembershipYears,0


In [12]:

products_df.isnull().sum().rename("Missing Values").to_frame()

,Missing Values
ProductID,0
Category,0
Brand,0
Price,0
Rating,15
PopularityScore,0


In [13]:
transactions_df.isnull().sum().rename("Missing Values").to_frame()

,Missing Values
CustomerID,0
ProductID,0
PurchaseCount,0
LastPurchaseDays,0
Spend,0


In [14]:
duplicate_count = customers_df.duplicated().sum()
print("Number of duplicate rows in customers_df:", duplicate_count)

Number of duplicate rows in customers_df: 0


In [15]:
duplicate_count = products_df.duplicated().sum()
print("Number of duplicate rows in products_df:", duplicate_count)

Number of duplicate rows in products_df: 0


In [16]:
duplicate_count = transactions_df.duplicated().sum()
print("Number of duplicate rows in transactions_df:", duplicate_count)

Number of duplicate rows in transactions_df: 384


In [17]:
transactions_df[transactions_df.duplicated(keep=False)].head(10)

,CustomerID,ProductID,PurchaseCount,LastPurchaseDays,Spend
4,4,64,3,354,1250.40
22,15,30,3,28,273.75
108,53,89,1,182,139.10
134,54,169,3,344,497.67
142,55,8,2,153,683.16
201,58,119,1,235,304.19
205,58,22,4,201,581.44
231,60,102,2,313,968.72
482,74,137,1,304,458.85
506,75,84,4,214,1042.20


In [18]:
duplicate_count = transactions_df.duplicated(
    subset=["CustomerID", "ProductID"]
).sum()

print("Number of duplicate CustomerID and ProductID combinations:", duplicate_count)

Number of duplicate CustomerID and ProductID combinations: 3474


In [19]:
# Fill missing ratings with the median rating
products_df["Rating"] = products_df["Rating"].fillna(products_df["Rating"].median())

In [20]:
merged_df = (
    transactions_df
    .merge(customers_df, on="CustomerID", how="left")
    .merge(products_df, on="ProductID", how="left")
)

print("merged_df shape:", merged_df.shape)
display(merged_df.head())

merged_df shape: (19166, 14)


,CustomerID,ProductID,PurchaseCount,LastPurchaseDays,Spend,AgeGroup,Region,LoyaltyLevel,MembershipYears,Category,Brand,Price,Rating,PopularityScore
0,1,82,3,162,104.01,36-50,East,Bronze,9,Books,HomePro,34.67,3.56,5
1,2,7,4,276,387.48,26-35,East,Bronze,4,Home,SmartBuy,96.87,3.79,49
2,3,170,4,215,1507.40,18-25,East,Silver,10,Kitchen,FitLife,376.85,2.78,4
3,3,9,1,251,341.40,18-25,East,Silver,10,Gaming,SmartBuy,341.40,4.23,3
4,4,64,3,354,1250.40,36-50,East,Bronze,14,Gaming,TechMax,416.80,3.59,1


In [21]:
missing_after_merge = merged_df.isnull().sum().rename("Missing Values").to_frame()
display(missing_after_merge)

,Missing Values
CustomerID,0
ProductID,0
PurchaseCount,0
LastPurchaseDays,0
Spend,0
AgeGroup,0
Region,0
LoyaltyLevel,0
MembershipYears,0
Category,0


In [22]:
top_10_customers = (
    merged_df.groupby("CustomerID", as_index=False)["Spend"]
    .sum()
    .rename(columns={"Spend": "TotalSpend"})
    .sort_values("TotalSpend", ascending=False)
    .head(10)
)

display(top_10_customers.style.hide(axis="index"))

CustomerID,TotalSpend
395,27323.950000
751,23084.090000
550,23065.130000
999,23012.880000
298,22883.980000
454,22750.970000
57,22713.060000
355,22533.650000
546,22399.850000
676,22273.500000


In [23]:
top_10_customers_by_frequency = (
    merged_df.groupby("CustomerID", as_index=False)["PurchaseCount"]
    .sum()
    .rename(columns={"PurchaseCount": "TotalPurchases"})
    .sort_values("TotalPurchases", ascending=False)
    .head(10)
)

display(top_10_customers_by_frequency.style.hide(axis="index"))

CustomerID,TotalPurchases
550,87
395,86
664,84
290,83
831,83
879,81
812,81
571,80
289,80
480,80


In [24]:
customer_value_analysis = (
    merged_df.groupby("CustomerID", as_index=False)
    .agg(
        TotalSpend=("Spend", "sum"),
        TotalPurchases=("PurchaseCount", "sum")
    )
)

customer_value_analysis["AverageSpendPerPurchase"] = (
    customer_value_analysis["TotalSpend"]
    / customer_value_analysis["TotalPurchases"]
)

top_10_customer_value = (
    customer_value_analysis
    .sort_values("TotalSpend", ascending=False)
    .head(10)
)

display(
    top_10_customer_value.style
    .hide(axis="index")
    .format({
        "TotalSpend": "{:.2f}",
        "AverageSpendPerPurchase": "{:.2f}"
    })
)

CustomerID,TotalSpend,TotalPurchases,AverageSpendPerPurchase
395,27323.95,86,317.72
751,23084.09,72,320.61
550,23065.13,87,265.12
999,23012.88,72,319.62
298,22883.98,76,301.11
454,22750.97,76,299.35
57,22713.06,71,319.90
355,22533.65,79,285.24
546,22399.85,79,283.54
676,22273.50,71,313.71


In [25]:
top_10_products = (
    merged_df.groupby(["ProductID", "Category", "Brand"], as_index=False)["Spend"]
    .sum()
    .rename(columns={"Spend": "TotalRevenue"})
    .sort_values("TotalRevenue", ascending=False)
    .head(10)
)

display(
    top_10_products.style
    .hide(axis="index")
    .format({"TotalRevenue": "{:.2f}"})
)

ProductID,Category,Brand,TotalRevenue
102,Books,FitLife,1756773.72
7,Home,SmartBuy,488127.93
41,Home,HomePro,446687.43
134,kitchen,SmartBuy,415775.14
8,Home,HomePro,240130.74
184,Fitness,SmartBuy,195388.13
45,Kitchen,NextGen,190771.02
65,Kitchen,TechMax,187267.92
81,Home,FitLife,182543.45
137,Electronics,NextGen,172986.45


In [26]:
revenue_by_category = (
    merged_df.groupby("Category", as_index=False)["Spend"]
    .sum()
    .rename(columns={"Spend": "TotalRevenue"})
    .sort_values("TotalRevenue", ascending=False)
)

display(
    revenue_by_category.style
    .hide(axis="index")
    .format({"TotalRevenue": "{:.2f}"})
)

Category,TotalRevenue
Books,2615588.29
Home,2335471.87
Electronics,1711481.42
Fitness,1339703.79
Gaming,1324502.70
Kitchen,1303202.15
kitchen,505228.12
fitness,241709.18
books,180158.11
gaming,87013.44


In [27]:
products_df["Category"] = products_df["Category"].str.strip().str.title()
merged_df["Category"] = merged_df["Category"].str.strip().str.title()

revenue_by_category = (
    merged_df.groupby("Category", as_index=False)["Spend"]
    .sum()
    .rename(columns={"Spend": "TotalRevenue"})
    .sort_values("TotalRevenue", ascending=False)
)

display(
    revenue_by_category.style
    .hide(axis="index")
    .format({"TotalRevenue": "{:.2f}"})
)

Category,TotalRevenue
Books,2795746.40
Home,2337945.87
Kitchen,1808430.27
Electronics,1796995.49
Fitness,1581412.97
Gaming,1411516.14


In [30]:
display(
    pd.DataFrame({"Brand": sorted(products_df["Brand"].dropna().unique())})
    .style.hide(axis="index")
)

Brand
FitLife
HomePro
NextGen
PrimeGoods
SmartBuy
TechMax


In [31]:
revenue_by_brand = (
    merged_df.groupby("Brand", as_index=False)["Spend"]
    .sum()
    .rename(columns={"Spend": "TotalRevenue"})
    .sort_values("TotalRevenue", ascending=False)
)

display(
    revenue_by_brand.style
    .hide(axis="index")
    .format({"TotalRevenue": "{:.2f}"})
)

Brand,TotalRevenue
FitLife,3348546.16
SmartBuy,2632371.13
HomePro,1911437.37
TechMax,1620948.69
NextGen,1306117.89
PrimeGoods,912625.90


In [32]:
region_category_spend = merged_df.pivot_table(
    index="Region",
    columns="Category",
    values="Spend",
    aggfunc="sum",
    fill_value=0
)

display(region_category_spend.style.format("{:.2f}"))

Category,Books,Electronics,Fitness,Gaming,Home,Kitchen
Region,,,,,,
East,691873.40,444412.28,387627.22,317068.51,587730.72,453168.15
North,670778.05,438791.06,400178.30,351333.24,559039.24,455125.77
South,692732.07,442415.28,365448.98,354521.38,598920.90,465457.01
West,740362.88,471376.87,428158.47,388593.01,592255.01,434679.34


In [33]:
loyalty_region_spend = merged_df.pivot_table(
    index="LoyaltyLevel",
    columns="Region",
    values="Spend",
    aggfunc="sum",
    fill_value=0
)

display(loyalty_region_spend.style.format("{:.2f}"))

Region,East,North,South,West
LoyaltyLevel,,,,
Bronze,1389136.43,1538040.32,1403325.54,1591579.09
Gold,561834.19,566922.55,687306.63,641141.13
Silver,930909.66,770282.79,828863.45,822705.36


In [34]:
interaction_matrix = merged_df.pivot_table(
    index="CustomerID",
    columns="ProductID",
    values="PurchaseCount",
    aggfunc="sum",
    fill_value=0
)

display(interaction_matrix.iloc[:5, :10])

ProductID,1,2,3,4,5,6,7,8,9,10
CustomerID,,,,,,,,,,
1,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,4,0,0,0
3,0,0,0,0,0,0,0,0,1,0
4,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0


In [37]:
top_10_popular_products = (
    merged_df.groupby("ProductID", as_index=False)["PurchaseCount"]
    .sum()
    .rename(columns={"PurchaseCount": "TotalPurchases"})
    .sort_values("TotalPurchases", ascending=False)
    .head(10)
    .merge(
        products_df[["ProductID", "Category", "Brand"]],
        on="ProductID",
        how="left"
    )[["ProductID", "Category", "Brand", "TotalPurchases"]]
)

display(top_10_popular_products.style.hide(axis="index"))

ProductID,Category,Brand,TotalPurchases
7,Home,SmartBuy,5039
102,Books,FitLife,3627
13,Electronics,SmartBuy,1837
134,Kitchen,SmartBuy,1397
41,Home,HomePro,1097
184,Fitness,SmartBuy,973
54,Electronics,PrimeGoods,862
195,Fitness,TechMax,825
8,Home,HomePro,703
130,Books,NextGen,580


In [38]:
customer_similarity = pd.DataFrame(
    cosine_similarity(interaction_matrix),
    index=interaction_matrix.index,
    columns=interaction_matrix.index
)

display(customer_similarity.iloc[:5, :5].style.format("{:.3f}"))

CustomerID,1,2,3,4,5
CustomerID,,,,,
1,1.000,0.000,0.000,0.000,0.000
2,0.000,1.000,0.000,0.000,0.000
3,0.000,0.000,1.000,0.000,0.000
4,0.000,0.000,0.000,1.000,0.000
5,0.000,0.000,0.000,0.000,1.000


In [40]:
def recommend_collaborative(customer_id, top_n=5):
    columns = ["ProductID", "Category", "Brand", "RecommendationScore"]

    if customer_id not in customer_similarity.index:
        return pd.DataFrame(columns=columns)

    similarities = customer_similarity.loc[customer_id].drop(customer_id)
    similarities = similarities[similarities > 0]

    if similarities.empty:
        return pd.DataFrame(columns=columns)

    scores = (
        interaction_matrix.loc[similarities.index]
        .mul(similarities, axis=0)
        .sum(axis=0)
    )

    purchased_products = interaction_matrix.loc[customer_id]
    scores = scores[purchased_products == 0]
    scores = scores[scores > 0]

    if scores.empty:
        return pd.DataFrame(columns=columns)

    recommendations = (
        scores.rename("RecommendationScore")
        .sort_values(ascending=False)
        .head(top_n)
        .rename_axis("ProductID")
        .reset_index()
        .merge(
            products_df[["ProductID", "Category", "Brand"]],
            on="ProductID",
            how="left"
        )[columns]
    )

    recommendations["RecommendationScore"] = recommendations[
        "RecommendationScore"
    ].round(3)

    return recommendations


recommend_collaborative(1)

,ProductID,Category,Brand,RecommendationScore
0,7,Home,SmartBuy,163.304
1,102,Books,FitLife,104.514
2,13,Electronics,SmartBuy,54.205
3,41,Home,HomePro,41.974
4,134,Kitchen,SmartBuy,37.594


In [41]:
for customer_id in [1, 2, 3, 4, 5]:
    print(f"CustomerID: {customer_id}")
    recommendations = recommend_collaborative(customer_id, top_n=5)
    display(
        recommendations.style
        .hide(axis="index")
        .format({"RecommendationScore": "{:.3f}"})
    )

CustomerID: 1


ProductID,Category,Brand,RecommendationScore
7,Home,SmartBuy,163.304
102,Books,FitLife,104.514
13,Electronics,SmartBuy,54.205
41,Home,HomePro,41.974
134,Kitchen,SmartBuy,37.594


CustomerID: 2


ProductID,Category,Brand,RecommendationScore
102,Books,FitLife,1211.089
13,Electronics,SmartBuy,584.094
134,Kitchen,SmartBuy,459.444
41,Home,HomePro,375.457
184,Fitness,SmartBuy,324.465


CustomerID: 3


ProductID,Category,Brand,RecommendationScore
7,Home,SmartBuy,133.465
102,Books,FitLife,105.365
13,Electronics,SmartBuy,63.583
134,Kitchen,SmartBuy,44.678
184,Fitness,SmartBuy,28.962


CustomerID: 4


ProductID,Category,Brand,RecommendationScore
7,Home,SmartBuy,413.873
13,Electronics,SmartBuy,161.453
134,Kitchen,SmartBuy,113.022
41,Home,HomePro,86.940
184,Fitness,SmartBuy,85.301


CustomerID: 5


ProductID,Category,Brand,RecommendationScore
7,Home,SmartBuy,103.884
102,Books,FitLife,90.753
13,Electronics,SmartBuy,42.323
134,Kitchen,SmartBuy,32.847
41,Home,HomePro,31.008


In [42]:
popularity_ranking = (
    merged_df.groupby("ProductID")["PurchaseCount"]
    .sum()
    .sort_values(ascending=False)
)

comparison = []

for customer in [1, 2, 3, 4, 5]:
    purchased = set(interaction_matrix.loc[customer][
        interaction_matrix.loc[customer] > 0
    ].index)

    popularity_recommendations = [
        product_id
        for product_id in popularity_ranking.index
        if product_id not in purchased
    ][:5]

    collaborative_recommendations = recommend_collaborative(
        customer, top_n=5
    )["ProductID"].tolist()

    comparison.append({
        "CustomerID": customer,
        "PopularityBased": popularity_recommendations,
        "CollaborativeFiltering": collaborative_recommendations
    })

comparison_df = pd.DataFrame(comparison)
display(comparison_df)

,CustomerID,PopularityBased,CollaborativeFiltering
0,1,"[7, 102, 13, 134, 41]","[7, 102, 13, 41, 134]"
1,2,"[102, 13, 134, 41, 184]","[102, 13, 134, 41, 184]"
2,3,"[7, 102, 13, 134, 41]","[7, 102, 13, 134, 184]"
3,4,"[7, 13, 134, 41, 184]","[7, 13, 134, 41, 184]"
4,5,"[7, 102, 13, 134, 41]","[7, 102, 13, 134, 41]"


In [44]:
rng = np.random.default_rng(42)

eligible_customers = interaction_matrix.index[
    (interaction_matrix > 0).sum(axis=1) >= 2
].tolist()

held_out_products = {
    customer_id: rng.choice(
        interaction_matrix.loc[customer_id][
            interaction_matrix.loc[customer_id] > 0
        ].index
    )
    for customer_id in eligible_customers
}

training_matrix = interaction_matrix.copy()

for customer_id, product_id in held_out_products.items():
    training_matrix.at[customer_id, product_id] = 0

popularity_ranking_eval = (
    training_matrix.sum(axis=0)
    .sort_values(ascending=False)
)

customer_similarity_eval = pd.DataFrame(
    cosine_similarity(training_matrix),
    index=training_matrix.index,
    columns=training_matrix.index
)

popularity_hits = 0
collaborative_hits = 0

for customer_id in eligible_customers:
    held_out_product = held_out_products[customer_id]
    purchased_products = training_matrix.loc[customer_id]
    purchased_products = purchased_products[purchased_products > 0].index

    popularity_recommendations_eval = [
        product_id
        for product_id in popularity_ranking_eval.index
        if product_id not in purchased_products
    ][:5]

    if held_out_product in popularity_recommendations_eval:
        popularity_hits += 1

    similarities = customer_similarity_eval.loc[customer_id].drop(customer_id)
    similarities = similarities[similarities > 0]

    if not similarities.empty:
        scores = (
            training_matrix.loc[similarities.index]
            .mul(similarities, axis=0)
            .sum(axis=0)
        )
        scores = scores.drop(index=purchased_products, errors="ignore")
        collaborative_recommendations_eval = (
            scores[scores > 0]
            .sort_values(ascending=False)
            .head(5)
            .index
            .tolist()
        )
    else:
        collaborative_recommendations_eval = []

    if held_out_product in collaborative_recommendations_eval:
        collaborative_hits += 1

evaluated_customers = len(eligible_customers)

evaluation_results = pd.DataFrame({
    "Method": [
        "Popularity-based",
        "User-based Collaborative Filtering"
    ],
    "HitRateAt5": [
        popularity_hits / evaluated_customers,
        collaborative_hits / evaluated_customers
    ]
})

print("Number of evaluated customers:", evaluated_customers)

display(
    evaluation_results.style
    .hide(axis="index")
    .format({"HitRateAt5": "{:.3f}"})
)

Number of evaluated customers: 971


Method,HitRateAt5
Popularity-based,0.205
User-based Collaborative Filtering,0.205
